# XGBoost Nutrient-Aware Pathogen Suppression Predictor 0201

In [14]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')


In [15]:

df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\community_index_v9.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nTarget (community_suppression) statistics:")
print(df['community_suppression'].describe().round(4))

print("\nRows per diversity level:")
print(df['diversity_level'].value_counts().sort_index())

print("\nRows per community size:")
print(df['community_size'].value_counts().sort_index())

Dataset shape: 48,000 rows × 168 columns

Target (community_suppression) statistics:
count    48000.0000
mean         0.6698
std          0.3767
min          0.0000
25%          0.4471
50%          0.8554
75%          0.9710
max          0.9990
Name: community_suppression, dtype: float64

Rows per diversity level:
diversity_level
1     15000
2     12000
4      9000
8      6000
16     6000
Name: count, dtype: int64

Rows per community size:
community_size
12    9600
16    9600
24    9600
32    9600
48    9600
Name: count, dtype: int64


In [16]:
# Species membership columns (100 binary features) 
# Each column is 1 if that species is present in the community, 0 otherwise.
species_cols = [f"Species_{i:04d}" for i in range(2, 102)]


# R0 nutrient concentration columns (20 features)
# R0_c1 through R0_c20: initial concentration of each carbon source.
# Most are 0 (inactive). Active ones are ~20/n_active
r0_cols = [f"R0_c{k}" for k in range(1, 21)]


# Active flags (20 binary features)
# active_c1 through active_c20: 1 if that carbon source is supplied, 0 if not.
# These are redundant with R0 (if R0>0 then active=1) but are kept anyways
# CLARIFY ONCE: DO WE NEED TO KEEP THIS?
active_cols = [f"active_c{k}" for k in range(1, 21)]

# Structural environment features (2 features)
# community_size:  how many species are in this community (12, 16, 24, 32, 48).
# diversity_level: how many carbon sources are active (1, 2, 4, 8, 16).

structural_cols = ['community_size', 'diversity_level']

# Combine into full feature matrix
# The + operator on lists concatenates them.
feature_cols = species_cols + r0_cols + active_cols + structural_cols
print(f"Total features: {len(feature_cols)}")
print(f"  Species membership:   {len(species_cols)}")
print(f"  R0 concentrations:    {len(r0_cols)}")
print(f"  Active flags:         {len(active_cols)}")
print(f"  Structural (size/div):{len(structural_cols)}")


X = df[feature_cols]
y = df['community_suppression']

print(f"\nX shape: {X.shape}    y shape: {y.shape}")
print(f"y range: [{y.min():.4f}, {y.max():.4f}]   mean: {y.mean():.4f}")

Total features: 142
  Species membership:   100
  R0 concentrations:    20
  Active flags:         20
  Structural (size/div):2

X shape: (48000, 142)    y shape: (48000,)
y range: [0.0000, 0.9990]   mean: 0.6698


In [17]:
# train test split

# We create a single stratification key that combines community_size and
# diversity_level. This ensures the 25 (size × diversity) combinations are
# equally represented in both train and test.
strat_key = df['community_size'].astype(str) + "_" + df['diversity_level'].astype(str)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test set:  {X_test.shape[0]:,} rows")
print(f"\nDiversity level distribution in test set:")

print(df.loc[X_test.index, 'diversity_level'].value_counts().sort_index())

Train set: 38,400 rows
Test set:  9,600 rows

Diversity level distribution in test set:
diversity_level
1     3000
2     2400
4     1800
8     1200
16    1200
Name: count, dtype: int64


In [18]:
# baseline models

print("Baseline comparisons")


# Baseline 1: Global mean predictor
# Always predicts y_mean. R²=0 by definition. 
y_mean = y_train.mean()

y_pred_mean = np.full_like(y_test, fill_value=y_mean, dtype=float)
r2_mean = r2_score(y_test, y_pred_mean)
print(f"Baseline (global mean predictor):  R² = {r2_mean:.4f}")

# Baseline 2: Ridge Regression 

from sklearn.linear_model import Ridge
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)

ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)
r2_ridge = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
print(f"Ridge Regression:                  R² = {r2_ridge:.4f}   MAE = {mae_ridge:.4f}")

Baseline comparisons
Baseline (global mean predictor):  R² = -0.0002
Ridge Regression:                  R² = 0.8531   MAE = 0.0988


In [19]:
# xgboost

model_init = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=30,
    verbosity=0
)


model_init.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_init = model_init.predict(X_test)
r2_init = r2_score(y_test, y_pred_init)
mae_init = mean_absolute_error(y_test, y_pred_init)
mse_init = mean_squared_error(y_test, y_pred_init)

print(f"XGBoost (initial):  R² = {r2_init:.4f}   MAE = {mae_init:.4f}   MSE = {mse_init:.5f}")
print(f"Best iteration: {model_init.best_iteration}")

XGBoost (initial):  R² = 0.9424   MAE = 0.0478   MSE = 0.00825
Best iteration: 499


In [7]:
# hyperparameter tuning

param_grid = {
    'max_depth':        [5, 6, 7],
    
    'learning_rate':    [0.01, 0.05,0.08, 0.1],
    
    'subsample':        [0.7, 0.8,0.9, 1.0],
    
    
    'colsample_bytree': [0.6, 0.8,0.9, 1.0],
   
}

# KFold(n_splits=3): splits training data into 3 folds.
# shuffle=True: randomises row order before splitting
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# base estimator with fixed params for gridsearch
# n_estimators low here to keep it fast, will retrain again with higher value once 
# best params found
base_estimator = xgb.XGBRegressor(
    n_estimators=200,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    verbosity=0
)

grid_search = GridSearchCV(
    estimator=base_estimator,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# .fit on a GridSearchCV trains ALL combinations and picks the best.
grid_search.fit(X_train, y_train)

print(f"\nBest parameters found:")

for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")


best_cv_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best CV RMSE: {best_cv_rmse:.4f}")

Fitting 3 folds for each of 192 candidates, totalling 576 fits

Best parameters found:
  colsample_bytree: 1.0
  learning_rate: 0.1
  max_depth: 7
  subsample: 0.8
Best CV RMSE: 0.0914


In [20]:
# final model with best params

best_params = grid_search.best_params_
print(best_params)

model_final = xgb.XGBRegressor(
    **best_params,
    n_estimators=1000,   # high ceiling; early stopping will find the right number
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=40,  # stop if no improvement for 40 rounds
    verbosity=0
)

model_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50   # print progress every 50 rounds
)

y_pred_final = model_final.predict(X_test)

r2_final  = r2_score(y_test, y_pred_final)
mae_final = mean_absolute_error(y_test, y_pred_final)
mse_final = mean_squared_error(y_test, y_pred_final)
rmse_final = np.sqrt(mse_final)

print(f"\n--- Final Model Performance ---")
print(f"R²:   {r2_final:.4f}   (1.0 = perfect)")
print(f"MAE:  {mae_final:.4f}  (mean absolute error in suppression units)")
print(f"RMSE: {rmse_final:.4f}")
print(f"MSE:  {mse_final:.5f}")
print(f"Best iteration (trees used): {model_final.best_iteration}")

# Compare all models together
print("\n--- Model Comparison ---")
print(f"{'Model':<30} {'R²':>8}  {'MAE':>8}")
print("-" * 48)
print(f"{'Global mean (baseline)':<30} {r2_mean:>8.4f}  {'N/A':>8}")
print(f"{'Ridge Regression':<30} {r2_ridge:>8.4f}  {mae_ridge:>8.4f}")
print(f"{'XGBoost (initial)':<30} {r2_init:>8.4f}  {mae_init:>8.4f}")
print(f"{'XGBoost (tuned)':<30} {r2_final:>8.4f}  {mae_final:>8.4f}")

{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'subsample': 0.8}
[0]	validation_0-rmse:0.34814
[50]	validation_0-rmse:0.11164
[100]	validation_0-rmse:0.09715
[150]	validation_0-rmse:0.09125
[200]	validation_0-rmse:0.08840
[250]	validation_0-rmse:0.08646
[300]	validation_0-rmse:0.08530
[350]	validation_0-rmse:0.08489
[400]	validation_0-rmse:0.08462
[450]	validation_0-rmse:0.08438
[500]	validation_0-rmse:0.08417
[550]	validation_0-rmse:0.08410
[600]	validation_0-rmse:0.08401
[632]	validation_0-rmse:0.08405

--- Final Model Performance ---
R²:   0.9508   (1.0 = perfect)
MAE:  0.0438  (mean absolute error in suppression units)
RMSE: 0.0840
MSE:  0.00705
Best iteration (trees used): 592

--- Model Comparison ---
Model                                R²       MAE
------------------------------------------------
Global mean (baseline)          -0.0002       N/A
Ridge Regression                 0.8531    0.0988
XGBoost (initial)                0.9424    0.0478
XGBoost (tuned)  

In [21]:
# per condition breakdown

results_df = pd.DataFrame({
    'y_true':         y_test.values,
    'y_pred':         y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

print("\nBy diversity level:")
print(f"{'D':<5} {'R²':>8}  {'MAE':>8}  {'N':>6}")
print("-" * 32)

for d, grp in results_df.groupby('diversity_level'):
    r2_d  = r2_score(grp['y_true'], grp['y_pred'])
    mae_d = mean_absolute_error(grp['y_true'], grp['y_pred'])
    print(f"D={d:<4} {r2_d:>8.4f}  {mae_d:>8.4f}  {len(grp):>6,}")

print("\nBy community size:")
print(f"{'N':<5} {'R²':>8}  {'MAE':>8}  {'Count':>6}")
print("-" * 32)
for n, grp in results_df.groupby('community_size'):
    r2_n  = r2_score(grp['y_true'], grp['y_pred'])
    mae_n = mean_absolute_error(grp['y_true'], grp['y_pred'])
    print(f"N={n:<4} {r2_n:>8.4f}  {mae_n:>8.4f}  {len(grp):>6,}")


By diversity level:
D           R²       MAE       N
--------------------------------
D=1      0.9483    0.0504   3,000
D=2      0.9552    0.0443   2,400
D=4      0.7787    0.0465   1,800
D=8      0.9345    0.0326   1,200
D=16     0.9278    0.0333   1,200

By community size:
N           R²       MAE   Count
--------------------------------
N=12     0.8884    0.0792   1,920
N=16     0.9232    0.0599   1,920
N=24     0.9674    0.0357   1,920
N=32     0.9742    0.0269   1,920
N=48     0.9912    0.0173   1,920


In [22]:
# built in feature importance


importances = model_final.feature_importances_

importance_series = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

print("\nTop 20 most important features (by gain):")
print(importance_series.head(20).round(5).to_string())

# Summarise importance by feature group
species_imp = importance_series[species_cols].sum()
r0_imp      = importance_series[r0_cols].sum()
active_imp  = importance_series[active_cols].sum()
struct_imp  = importance_series[structural_cols].sum()

print(f"\nImportance grouped by feature type (total):")
print(f"  Species membership:  {species_imp:.4f}  ({100*species_imp:.1f}%)")
print(f"  R0 concentrations:   {r0_imp:.4f}  ({100*r0_imp:.1f}%)")
print(f"  Active flags:        {active_imp:.4f}  ({100*active_imp:.1f}%)")
print(f"  Structural (size/div):{struct_imp:.4f}  ({100*struct_imp:.1f}%)")


Top 20 most important features (by gain):
R0_c4              0.46158
R0_c17             0.22977
R0_c11             0.17747
R0_c1              0.03523
R0_c8              0.00897
R0_c7              0.00868
R0_c19             0.00668
R0_c15             0.00458
R0_c3              0.00348
diversity_level    0.00245
R0_c2              0.00221
R0_c14             0.00205
R0_c16             0.00201
Species_0072       0.00186
Species_0064       0.00177
Species_0071       0.00155
Species_0101       0.00149
Species_0034       0.00147
Species_0086       0.00135
community_size     0.00134

Importance grouped by feature type (total):
  Species membership:  0.0468  (4.7%)
  R0 concentrations:   0.9494  (94.9%)
  Active flags:        0.0000  (0.0%)
  Structural (size/div):0.0038  (0.4%)


In [23]:
# shapley value analysis

print('shap analysis')

explainer = shap.TreeExplainer(model_final)

sample_idx = np.random.default_rng(42).choice(len(X_test), size=2000, replace=False)

X_sample = X_test.iloc[sample_idx]


shap_values = explainer.shap_values(X_sample)

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

print("\nTop 20 features by mean |SHAP| value:")
print(shap_importance.head(20).round(5).to_string())

# SHAP by feature group
sp_shap     = shap_importance[species_cols].sum()
r0_shap     = shap_importance[r0_cols].sum()
act_shap    = shap_importance[active_cols].sum()
struct_shap = shap_importance[structural_cols].sum()
total_shap  = sp_shap + r0_shap + act_shap + struct_shap

print(f"\nSHAP importance by feature group (sum of mean |SHAP|):")
print(f"  Species membership:  {sp_shap:.4f}  ({100*sp_shap/total_shap:.1f}%)")
print(f"  R0 concentrations:   {r0_shap:.4f}  ({100*r0_shap/total_shap:.1f}%)")
print(f"  Active flags:        {act_shap:.4f}  ({100*act_shap/total_shap:.1f}%)")
print(f"  Structural:          {struct_shap:.4f}  ({100*struct_shap/total_shap:.1f}%)")

shap analysis

Top 20 features by mean |SHAP| value:
R0_c4             0.11743
R0_c1             0.05364
R0_c11            0.05041
R0_c17            0.04628
community_size    0.02986
R0_c19            0.02334
Species_0064      0.01640
Species_0055      0.01602
R0_c8             0.01489
Species_0071      0.01419
Species_0084      0.01341
R0_c16            0.01266
Species_0086      0.01261
R0_c15            0.01204
R0_c3             0.01064
Species_0053      0.01034
R0_c14            0.01010
R0_c9             0.00909
R0_c12            0.00906
R0_c7             0.00888

SHAP importance by feature group (sum of mean |SHAP|):
  Species membership:  0.2669  (37.8%)
  R0 concentrations:   0.4034  (57.1%)
  Active flags:        0.0000  (0.0%)
  Structural:          0.0358  (5.1%)


In [24]:
# figures

print("\n" + "=" * 60)
print("SECTION 12: Generating plots → suppression_results.png")
print("=" * 60)

# plt.figure(figsize=(width, height)): creates a new figure window.
# figsize is in inches. dpi (dots per inch) controls resolution.
fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle('XGBoost Nutrient-Aware Suppression Predictor — MiCRM v8',
             fontsize=15, fontweight='bold', y=0.98)

# plt.subplot(nrows, ncols, panel_index): creates a subplot in a grid.
# Here we use a 2×3 grid (2 rows, 3 columns), panels numbered 1–6.

# ---- Panel 1: Predicted vs Actual ----
ax1 = fig.add_subplot(2, 3, 1)
# ax.scatter(x, y): plots individual data points.
# alpha=0.3 makes points semi-transparent so density is visible.
# c sets colour, s sets marker size.
ax1.scatter(y_test, y_pred_final, alpha=0.15, s=3, c='steelblue')
# plot a y=x reference line (perfect predictions would lie on this line)
ax1.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Perfect prediction')
ax1.set_xlabel('True community suppression')
ax1.set_ylabel('Predicted community suppression')
ax1.set_title(f'Predicted vs Actual  (R²={r2_final:.3f})')
ax1.legend(fontsize=8)
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(-0.05, 1.05)

# ---- Panel 2: Residuals ----
# residuals = y_true - y_pred
# Ideally should be random noise centred at 0.
# Systematic patterns (e.g. always under-predicting at high suppression)
# suggest the model is missing something.
residuals = y_test.values - y_pred_final
ax2 = fig.add_subplot(2, 3, 2)
ax2.scatter(y_pred_final, residuals, alpha=0.15, s=3, c='darkorange')
ax2.axhline(0, color='r', linestyle='--', linewidth=1.5)
# np.percentile(array, [5, 95]): returns the 5th and 95th percentile values.
p5, p95 = np.percentile(residuals, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', linewidth=1, label=f'5th pct: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', linewidth=1, label=f'95th pct: {p95:.3f}')
ax2.set_xlabel('Predicted suppression')
ax2.set_ylabel('Residual (true − predicted)')
ax2.set_title('Residual Plot')
ax2.legend(fontsize=7)

# ---- Panel 3: Top 20 SHAP features ----
ax3 = fig.add_subplot(2, 3, 3)
top20_shap = shap_importance.head(20)
colours = []
for feat in top20_shap.index:
    if feat in species_cols:  colours.append('#2196F3')   # blue = species
    elif feat in r0_cols:     colours.append('#4CAF50')   # green = R0
    elif feat in active_cols: colours.append('#FF9800')   # orange = active
    else:                     colours.append('#9C27B0')   # purple = structural
# barh: horizontal bar chart. top20_shap.values = bar lengths.
# [::-1] reverses so highest is at top.
ax3.barh(range(20), top20_shap.values[::-1], color=colours[::-1])
ax3.set_yticks(range(20))
ax3.set_yticklabels(top20_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP value|')
ax3.set_title('Top 20 Features (SHAP)')

# Add a legend for colour coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='Species'),
    Patch(facecolor='#4CAF50', label='R0'),
    Patch(facecolor='#FF9800', label='Active'),
    Patch(facecolor='#9C27B0', label='Structural'),
]
ax3.legend(handles=legend_elements, loc='lower right', fontsize=7)

# ---- Panel 4: R² by diversity level ----
ax4 = fig.add_subplot(2, 3, 4)
div_levels = sorted(results_df['diversity_level'].unique())
r2_by_div = [r2_score(
    results_df[results_df['diversity_level']==d]['y_true'],
    results_df[results_df['diversity_level']==d]['y_pred']
) for d in div_levels]
ax4.bar([str(d) for d in div_levels], r2_by_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', linewidth=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level (# active nutrients)')
ax4.set_ylabel('R²')
ax4.set_title('Model R² by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)

# ---- Panel 5: R² by community size ----
ax5 = fig.add_subplot(2, 3, 5)
sizes = sorted(results_df['community_size'].unique())
r2_by_size = [r2_score(
    results_df[results_df['community_size']==n]['y_true'],
    results_df[results_df['community_size']==n]['y_pred']
) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_by_size, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', linewidth=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N')
ax5.set_ylabel('R²')
ax5.set_title('Model R² by Community Size')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)

# ---- Panel 6: SHAP summary by feature group (pie) ----
ax6 = fig.add_subplot(2, 3, 6)
group_names   = ['Species\nmembership', 'R0\nconcentrations', 'Active\nflags', 'Structural']
group_shap    = [sp_shap, r0_shap, act_shap, struct_shap]
group_colours = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
# ax.pie: creates a pie chart.
# autopct='%.1f%%': formats each slice's label as a percentage with 1 decimal.
# startangle: rotates the starting position.
wedges, texts, autotexts = ax6.pie(
    group_shap, labels=group_names, colors=group_colours,
    autopct='%.1f%%', startangle=90, textprops={'fontsize': 9}
)
ax6.set_title('SHAP Importance by Feature Group')

plt.tight_layout(rect=[0, 0, 1, 0.97])
# tight_layout: automatically adjusts subplot spacing to avoid overlap.
# rect=[left, bottom, right, top] reserves space for the suptitle.

plt.savefig('suppression_results_updated02.png', dpi=150, bbox_inches='tight')
# bbox_inches='tight': crops the saved figure to the actual content boundary
# (removes whitespace margins). Always use this.
plt.close()
# plt.close(): frees memory by closing the figure after saving.

print("Saved: suppression_results_updated02.png")


SECTION 12: Generating plots → suppression_results.png
Saved: suppression_results_updated02.png


In [ ]:
# =============================================================================
# SECTION 13 — SAVE MODEL & PREDICTIONS
# =============================================================================

print("\n" + "=" * 60)
print("SECTION 13: Saving outputs")
print("=" * 60)

# Save the model in XGBoost's native JSON format.
# This preserves all tree structures and can be reloaded with:
#   loaded_model = xgb.XGBRegressor()
#   loaded_model.load_model('xgb_model_v8.json')
model_final.save_model('xgb_model_v8.json')
print("Model saved: xgb_model_v8.json")

# Save predictions alongside true values for downstream analysis.
pred_df = pd.DataFrame({
    'community_id':       df.loc[X_test.index, 'community_id'].values,
    'broth_id':           df.loc[X_test.index, 'broth_id'].values,
    'diversity_level':    df.loc[X_test.index, 'diversity_level'].values,
    'community_size':     df.loc[X_test.index, 'community_size'].values,
    'y_true':             y_test.values,
    'y_pred':             y_pred_final,
    'residual':           y_test.values - y_pred_final,
})
pred_df.to_csv('predictions_v8.csv', index=False)
print("Predictions saved: predictions_v8.csv")

# Save SHAP values for the 2000-sample subset.
shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df.to_csv('shap_values_v8.csv', index=False)
print("SHAP values saved: shap_values_v8.csv")